# 🔥 BurnSight — Colab 실행 가이드

**순서대로 셀을 실행하세요.** 데이터가 없어도 Step 3에서 합성 데이터로 바로 시작할 수 있습니다.

> ⚡ 먼저: `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후 진행

---

| 데이터 옵션 | 대상 | 방법 |
|-------------|------|------|
| **A. 합성 데이터 (데모)** | 누구나 | Step 3에서 자동 생성 — 승인 불필요 |
| **B. 익명화 샘플** | 연구자 | `your-email@institution.edu` 로 이메일 요청 |
| **C. 전체 학습 데이터** | 협력 기관 | 데이터 공유 협약 후 제공 |

> ⚠️ 실제 환자 이미지는 개인정보 보호로 인해 공개하지 않습니다.

## Step 1 — 저장소 클론

In [ ]:
GITHUB_URL = "https://github.com/cjkmj16/BurnSight.git"

import os
!git clone {GITHUB_URL} /content/BurnSight
os.chdir("/content/BurnSight/BurnSight_src")
print("작업 디렉터리:", os.getcwd())

## Step 2 — 패키지 설치

In [ ]:
%%capture
!pip install -q opencv-python-headless scikit-image imageio h5py joblib albumentations
# tensorflow / numpy / sklearn / scipy / matplotlib 은 Colab에 기본 설치됨

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## Step 3 — 데이터 준비

`DATA_MODE`를 설정하고, 아래 **해당하는 셀만** 실행하세요.

In [ ]:
# ✏️  'synthetic' / 'sample' / 'full' 중 선택
DATA_MODE = "synthetic"
print("데이터 모드:", DATA_MODE)

### Step 3-A　합성 데이터 생성 (DATA_MODE = 'synthetic')
승인 없이 전체 파이프라인을 테스트할 수 있는 6프레임 합성 상처 이미지를 생성합니다.  
실제 데이터를 사용한다면 이 셀을 **건너뛰세요**.

In [ ]:
import os, numpy as np, cv2, matplotlib.pyplot as plt

if DATA_MODE != "synthetic":
    print("건너뜀 — DATA_MODE가 'synthetic'이 아닙니다.")
else:
    SYNTHETIC_DIR = "/content/synthetic_test"
    os.makedirs(SYNTHETIC_DIR, exist_ok=True)
    np.random.seed(42)
    T, H, W = 6, 128, 128

    def make_wound_frame(t, T, H, W):
        img = np.ones((H, W, 3), np.float32) * np.array([0.82, 0.70, 0.60])  # 피부색
        heal = t / max(T - 1, 1)          # 0(초기) → 1(치유)
        cx, cy = W // 2, H // 2
        rx = int(W * 0.28 * (1 - heal * 0.35))
        ry = int(H * 0.22 * (1 - heal * 0.35))
        yy, xx = np.ogrid[:H, :W]
        wound = ((xx-cx)/rx)**2 + ((yy-cy)/ry)**2 <= 1.0
        img[wound] = [0.75 + 0.20*(1-heal), 0.25 + 0.30*heal, 0.25 + 0.25*heal]
        erx, ery = int(rx*1.15), int(ry*1.15)
        ring = (((xx-cx)/erx)**2 + ((yy-cy)/ery)**2 <= 1.0) & ~wound
        fade = max(0.0, 1.0 - heal * 1.8)
        img[ring] = img[ring] * (1 - fade*0.4) + np.array([0.30,0.20,0.15]) * fade*0.4
        img = np.clip(img + np.random.normal(0, 0.015, img.shape).astype(np.float32), 0, 1)
        return (img * 255).astype(np.uint8)

    for t in range(T):
        frame = make_wound_frame(t, T, H, W)
        cv2.imwrite(os.path.join(SYNTHETIC_DIR, f"Day{t+1}_2024010{t+1}.png"),
                    cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    fig, axes = plt.subplots(1, T, figsize=(3*T, 3))
    for i, ax in enumerate(axes):
        img = cv2.cvtColor(cv2.imread(os.path.join(SYNTHETIC_DIR, f"Day{i+1}_2024010{i+1}.png")), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.set_title(f"Day {i+1}"); ax.axis("off")
    plt.suptitle("합성 상처 시퀀스 (데모용 — 실제 환자 이미지 아님)", fontsize=11)
    plt.tight_layout(); plt.show()

    TEST_DIR = SYNTHETIC_DIR
    print("\n✅ 합성 데이터 준비 완료:", TEST_DIR)

### Step 3-B/C　실제 데이터 (DATA_MODE = 'sample' 또는 'full')
접근 승인을 받은 경우에만 실행하세요.

In [ ]:
if DATA_MODE == "synthetic":
    print("건너뜀 — 합성 모드입니다.")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive"

    # ✏️  승인받은 Drive 폴더 경로로 수정
    TEST_DIR         = f"{DRIVE_ROOT}/BurnSight_sample"
    MASK_MODEL_PATH  = f"{DRIVE_ROOT}/mask_model.keras"
    CREATOR_PATH     = f"{DRIVE_ROOT}/creator.keras"

    for label, path in [("TEST_DIR", TEST_DIR), ("MASK_MODEL_PATH", MASK_MODEL_PATH), ("CREATOR_PATH", CREATOR_PATH)]:
        print(("✅" if os.path.exists(path) else "❌ 없음"), label, ":", path)

## Step 4 — config.py 경로 패치

In [ ]:
import os
from pathlib import Path

CACHE_DIR = "/content/cache_dir"
os.makedirs(CACHE_DIR, exist_ok=True)

# 합성 모드에서는 TEST_DIR만 필요; 나머지 경로는 더미
BASE_DIR    = TEST_DIR if DATA_MODE == "synthetic" else TEST_DIR
AUG_DIR     = TEST_DIR
ORIGIN_DIR  = TEST_DIR

CONFIG = f'''
import os, warnings, random
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras.mixed_precision import set_global_policy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from joblib import Memory

set_global_policy("float32")
warnings.filterwarnings("ignore", category=DeprecationWarning)

base_dir          = r"{BASE_DIR}"
original_test_dir = r"{TEST_DIR}"
cache_dir         = r"{CACHE_DIR}"
ORIGIN_DIR        = r"{ORIGIN_DIR}"
AUG_DIR           = r"{AUG_DIR}"
memory            = Memory(cache_dir, verbose=0)

physical_devices = tf.config.experimental.list_physical_devices("GPU")
if physical_devices:
    try: tf.config.experimental.set_memory_growth(physical_devices[0], True)
    except: pass

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED); tf.random.set_seed(SEED); random.seed(SEED)

SEQLEN = SEQ_LEN = sequence_length = 6
IMG_SIZE = (64, 64); IMG_H = IMG_W = 64
BATCH = 8; VAL_RATIO = 0.10; MAX_T = 12; STRIDE = 1
NUM_CLASSES = 5; WOUND_IDX = 1; ESCHAR_IDX = 2; HEALED_IDX = 3; EXCLUDE_IDX = 4
LESION_IDXS = (1, 2, 3); CHANGE_IDXS = (1, 2); STABLE_IDXS = (0, 3); POLICY_VER = 3
K_MIN = 0; K_MAX = 5; K_VAL = 5
THR_WOUND = 0.55; THR_SUPPORT = 0.45; THR_CHANGE = 0.40
LATENT_DIM = 256; CTX_K = 6; DELTA_MIN = 1; DELTA_MAX = 5
TAU_NCE = 0.1; TAU_ANTI = 0.03; PATCH_K = 32; TOPK_RATIO = 0.3
LR_STAGE1 = 1e-4; LR_STAGE2 = 1e-4
AUTOTUNE = tf.data.AUTOTUNE
input_shape = (None, 64, 64, 3); sequence_tensor = tf.constant(5); batch_size = 4
ckpt    = ModelCheckpoint("creator_model.keras", monitor="val_cos", save_best_only=True, mode="max", verbose=1)
early   = EarlyStopping(monitor="val_cos", patience=20, mode="max", restore_best_weights=True)
plateau = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5, verbose=1)
DEBUG_ROOT = Path(cache_dir) / "debug"; DEBUG_ROOT.mkdir(parents=True, exist_ok=True)
debug_dir  = DEBUG_ROOT / "dbg_inverted"; debug_dir.mkdir(exist_ok=True)
DBG_LIST   = debug_dir / "inverted_list.txt"
def log_inverted(tag, img_path, mask_path):
    with open(DBG_LIST, "a") as f: f.write(f"{{tag}}\\t{{img_path}}\\t{{mask_path}}\\n")
def _to_str(x):
    x = x.numpy() if hasattr(x, "numpy") else x
    return x.decode("utf-8", "ignore") if isinstance(x, (bytes, np.bytes_)) else str(x)
'''

with open("src/config.py", "w") as f:
    f.write(CONFIG)
print("✅ src/config.py 패치 완료")

## Step 5 — 사전 학습 모델 로드

In [ ]:
import tensorflow as tf, sys
sys.path.insert(0, "/content/BurnSight/BurnSight_src")

from src.config import *
from src.data.file_utils import *
from src.models.layers import *
from src.models.unet import *
from src.utils.image_utils import *
from src.inference.mask_utils import *
from src.inference.postprocess import *

if DATA_MODE == "synthetic":
    print("⚠️  합성 모드: 모델 없이 파이프라인 구조만 확인합니다.")
    print("   실제 추론을 하려면 모델 접근 신청 후 DATA_MODE를 변경하세요.")
    mask_model = None
    creator    = None
else:
    mask_model = tf.keras.models.load_model(MASK_MODEL_PATH)
    creator    = tf.keras.models.load_model(CREATOR_PATH)
    print("✅ mask_model:", mask_model.input_shape)
    print("✅ creator:   ", creator.input_shape)

## Step 6 — 테스트 시퀀스 로드

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt

sorted_image_files = get_sorted_day_images(TEST_DIR)
print(f"이미지 {len(sorted_image_files)}장 발견")

example_sequence = []
for path in sorted_image_files:
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (64, 64)).astype(np.float32) / 255.0
    example_sequence.append(img)

example_sequence = np.asarray(example_sequence, dtype=np.float32)[None, ...]  # (1,T,64,64,3)

T = example_sequence.shape[1]
if T > SEQLEN:
    example_sequence = example_sequence[:, -SEQLEN:]
elif T < SEQLEN:
    pad = np.repeat(example_sequence[:, :1], SEQLEN - T, axis=1)
    example_sequence = np.concatenate([pad, example_sequence], axis=1)

print("시퀀스 shape:", example_sequence.shape)

fig, axes = plt.subplots(1, SEQLEN, figsize=(3*SEQLEN, 3))
for t, ax in enumerate(axes):
    ax.imshow(example_sequence[0, t]); ax.set_title(f"t={t}"); ax.axis("off")
plt.suptitle("입력 시퀀스"); plt.tight_layout(); plt.show()

## Step 7 — ROI 게이트 생성 (m_soft)
*합성 모드에서는 모델 없이 더미 ROI를 사용합니다.*

In [ ]:
if mask_model is None:
    # 합성 모드: 중앙 타원형 더미 ROI
    B, T, H, W, _ = example_sequence.shape
    yy, xx = np.ogrid[:H, :W]
    ellipse = ((xx - W//2)/(W*0.28))**2 + ((yy - H//2)/(H*0.22))**2 <= 1.0
    m_soft = np.tile(ellipse.astype(np.float32)[None,None,...,None], (B,T,1,1,1))
    m_bin  = (m_soft > 0.5).astype(np.float32)
    print("더미 ROI 생성 (합성 모드)")
else:
    def preprocess_infer_seq(x): return x * 2.0 - 1.0
    m_soft, m_bin, roi_info = make_fixed_msoft_from_seq(
        example_sequence, mask_model, preprocess=preprocess_infer_seq,
        THR=THR_WOUND, mode="mixed", tau=0.30, ema_alpha=0.4,
        temp_T=0.9, use_tta=True, thr_scope="sequence", thr_reduce="median",
    )
    print("ROI info:", roi_info)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(example_sequence[0,-1]);              axes[0].set_title("마지막 프레임");  axes[0].axis("off")
axes[1].imshow(m_soft[0,-1,...,0], cmap="hot");      axes[1].set_title("m_soft (ROI)"); axes[1].axis("off")
axes[2].imshow(m_bin[0,-1,...,0],  cmap="gray");     axes[2].set_title("m_bin (이진)"); axes[2].axis("off")
plt.tight_layout(); plt.show()

## Step 8 — K-sweep 추론
K = 0.2 / 0.6 / 1.0 에서 치유 예측 이미지를 생성합니다.  
*합성 모드에서는 creator가 없으므로 K-채널 합성 시각화만 보여줍니다.*

In [ ]:
K_vals = [0.2, 0.6, 1.0]
results = {}

last01 = example_sequence[0, -1]  # (64,64,3)

for k in K_vals:
    if creator is None:
        # 합성 모드: ROI 안에서 K에 비례해 밝기 조정 (시각화 목적)
        pred = last01.copy()
        roi2d = m_soft[0, -1, ..., 0] > 0.5
        pred[roi2d] = np.clip(pred[roi2d] * (1 - k * 0.25) + k * 0.08, 0, 1)
    else:
        rgb_m11 = example_sequence * 2.0 - 1.0
        k_map   = np.full(m_soft.shape, k / float(K_MAX), np.float32)
        Xk      = np.concatenate([rgb_m11, k_map, m_soft], axis=-1)
        y_pred, x_last = creator.predict(Xk, verbose=0)
        pred    = np.clip((y_pred[0] + 1.0) / 2.0, 0, 1).astype(np.float32)
        last01  = np.clip((x_last[0] + 1.0) / 2.0, 0, 1).astype(np.float32)

    results[k] = pred

fig, axes = plt.subplots(1, len(K_vals) + 1, figsize=(4*(len(K_vals)+1), 4))
axes[0].imshow(last01); axes[0].set_title("현재 (마지막 프레임)"); axes[0].axis("off")
for i, k in enumerate(K_vals):
    axes[i+1].imshow(results[k]); axes[i+1].set_title(f"예측  K={k}"); axes[i+1].axis("off")
label = "(더미 시각화 — 실제 모델 없음)" if creator is None else ""
plt.suptitle(f"상처 치유 궤적 예측 {label}", fontsize=12)
plt.tight_layout(); plt.show()

## Step 9 — Lab 색상 변화 분석 (Δa*, ΔE)

In [ ]:
import pandas as pd, cv2

def rgb01_to_lab(img01):
    x8 = (np.clip(img01, 0, 1) * 255).astype(np.uint8)
    return cv2.cvtColor(x8, cv2.COLOR_RGB2LAB).astype(np.float32)

roi_w = m_soft[0, -1, ..., 0]  # (64,64)
rows = []
for k in K_vals:
    Lab_l = rgb01_to_lab(last01)
    Lab_p = rgb01_to_lab(results[k])
    d = Lab_p - Lab_l
    dE = np.sqrt(np.sum(d**2, axis=-1))
    w  = np.clip(roi_w, 0, 1)
    wsum = w.sum() + 1e-6
    rows.append({"K": k,
                 "Δa*_ROI (적색도 변화)": round(float((d[...,1]*w).sum()/wsum), 4),
                 "ΔE_ROI (전체 색변화)": round(float((dE*w).sum()/wsum), 4)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(df["K"], df["Δa*_ROI (적색도 변화)"], "o-")
axes[0].axhline(0, ls="--", color="gray")
axes[0].set_xlabel("K"); axes[0].set_title("Δa* (적색도 변화) vs K")
axes[1].plot(df["K"], df["ΔE_ROI (전체 색변화)"], "s-", color="orange")
axes[1].set_xlabel("K"); axes[1].set_title("ΔE vs K")
plt.tight_layout(); plt.show()

---
## ✅ 완료!

전체 파이프라인을 실행했습니다:
- Step 1–2: 저장소 클론 + 패키지 설치
- Step 3: 데이터 준비 (합성 or 실제)
- Step 4: config 자동 패치
- Step 5: 모델 로드
- Step 6–7: 시퀀스 로드 + ROI 게이트
- Step 8–9: K-sweep 추론 + 색상 분석

**실제 모델로 추론하려면** → `your-email@institution.edu`로 접근 신청 후 `DATA_MODE`를 `'sample'`로 변경하세요.